# Vertical Chat
A sample how to build a chat for small business using:

* GPT 35
* Panel
* OpenAI


This is just a simple sample to start to understand how the OpenAI API works, and how to create Prompts. It Is really far from beign a complete solution.
We are going to introduce some interesting points:

* The roles in a conversation.
* How is the conversations’ memory preserved?

Deeper explanations in the article: [Create Your First Chatbot Using GPT 3.5, OpenAI, Python and Panel.](https://medium.com/towards-artificial-intelligence/create-your-first-chatbot-using-gpt-3-5-openai-python-and-panel-7ec180b9d7f2)

In [1]:
#if you need an API Key from OpenAI
#https://platform.openai.com/account/api-keys

from openai import OpenAI
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')

In [2]:
client = OpenAI(
    # This is the default and can be omitted
    api_key=OPENAI_API_KEY,
)

def continue_conversation(messages, temperature=0):
    response = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=messages,
        temperature=temperature,
    )
    #print(str(response.choices[0].message["content"]))
    return response.choices[0].message.content

In [3]:
def add_prompts_conversation(_):
    #Get the value introduced by the user
    prompt = client_prompt.value_input
    client_prompt.value = ''

    #Append to the context the User prompt.
    context.append({'role':'user', 'content':f"{prompt}"})

    #Get the response.
    response = continue_conversation(context)

    #Add the response to the context.
    context.append({'role':'assistant', 'content':f"{response}"})

    #Update the panels to show the conversation.
    panels.append(
        pn.Row('User:', pn.pane.Markdown(prompt, width=600)))
    panels.append(
        pn.Row('Assistant:', pn.pane.Markdown(response, width=600)))

    return pn.Column(*panels)

In [4]:
import ipywidgets as widgets
from IPython.display import display, Markdown, clear_output

# ============================================================
# 3 PROMPT VARIATIONS
# ============================================================

PROMPT_V1_FRANKFURT = """
Act as an OrderBot, you work collecting orders in a delivery only fast food restaurant called
My Dear Frankfurt. First welcome the customer in a very friendly way, then collect the order.
Wait to collect the entire order, beverages included, then summarize it and check for a final
time if everything is ok or the customer wants to add anything else. Finally collect the payment.
Make sure to clarify all options, extras and sizes to uniquely identify the item from the menu.
You respond in a short, very friendly style.
Menu:
burger 12.95, 10.00, 7.00
frankfurt 10.95, 9.25, 6.50
sandwich 11.95, 9.75, 6.75
fries 4.50, 3.50
salad 7.25
Toppings: extra cheese 2.00, mushrooms 1.50, martra sausage 3.00, canadian bacon 3.50,
romesco sauce 1.50, peppers 1.00
Drinks: coke 3.00, 2.00, 1.00; sprite 3.00, 2.00, 1.00; vichy catalan 5.00
"""

PROMPT_V2_PIZZERIA = """
You are PizzaBot, a warm Italian-style assistant taking delivery orders for "Bella Napoli Pizzeria".
Greet the customer with an Italian flair (e.g. "Ciao!"). Ask what they would like, ALWAYS confirm
size (Small / Medium / Large) and crust type (Thin / Classic / Stuffed). Suggest a drink if they
forget one. After the order is complete, repeat it back, ask for the delivery address, then provide
a total and ask for payment method (Cash / Card).
IMPORTANT RULES:
- Never invent menu items. If asked for something not on the menu, politely say it is unavailable.
- All prices are in EUR.
Menu (Small / Medium / Large):
Margherita 8 / 11 / 14
Pepperoni 9 / 12 / 15
Quattro Formaggi 10 / 13 / 16
Diavola 9.5 / 12.5 / 15.5
Vegetariana 9 / 12 / 15
Pasta:
Spaghetti Carbonara 11
Penne Arrabbiata 10
Lasagna 12
Drinks:
Coke 2.5 | Sparkling Water 2 | House Red Wine (glass) 5 | Birra Moretti 4
"""

PROMPT_V3_SUSHI = """
You are SushiSan, a polite ordering assistant for "Sakura Sushi Tokyo", a Japanese delivery place.
Tone: respectful, calm, sprinkle Japanese words (Irasshaimase!, Hai!, Arigatou!).
Workflow:
1. Greet warmly.
2. Ask if the customer wants Individual pieces, Combo sets, or Chef's omakase recommendation.
3. Always confirm spice level for spicy items (Mild / Medium / Hot).
4. Ask about allergies (especially soy, sesame, shellfish) BEFORE finalizing.
5. Summarize order, ask delivery time preference, then collect payment.
STRICT RULES:
- Stay strictly within the menu below — never invent rolls or prices.
- Quote prices in USD.
Menu:
Nigiri (2 pcs): Salmon 5, Tuna 6, Eel 7, Shrimp 5
Maki rolls (8 pcs): California 8, Spicy Tuna 10, Dragon 14, Rainbow 15
Combo sets: Sakura Set (12 pcs) 22, Tokyo Deluxe (18 pcs) 32
Sides: Miso Soup 3, Edamame 4, Seaweed Salad 5
Drinks: Green Tea 2, Ramune 3, Sake (small) 6, Asahi Beer 5
"""

PROMPTS = {
    "V1 · Frankfurt 🌭": PROMPT_V1_FRANKFURT,
    "V2 · Pizzeria 🍕":  PROMPT_V2_PIZZERIA,
    "V3 · Sushi 🍣":     PROMPT_V3_SUSHI,
}

# Active conversation context (mutable — replaced when version is switched)
context = [{'role': 'system', 'content': PROMPT_V1_FRANKFURT}]
current_version = {"name": "V1 · Frankfurt 🌭"}

# ============================================================
# GUI WIDGETS
# ============================================================
title       = widgets.HTML("<h2>🤖 Multi-Prompt OrderBot Lab</h2>")
status      = widgets.HTML(f"<b>Active prompt:</b> {current_version['name']}")
client_in   = widgets.Text(value='Hi', placeholder='Type your message…',
                           description='You:', layout=widgets.Layout(width='600px'))
talk_btn    = widgets.Button(description='Talk',  button_style='primary', icon='paper-plane')
close_btn   = widgets.Button(description='Close', button_style='danger',  icon='times')

v_buttons = [widgets.Button(description=name, button_style='info',
                            layout=widgets.Layout(width='180px'))
             for name in PROMPTS.keys()]

chat_out = widgets.Output(layout=widgets.Layout(border='1px solid #ddd',
                                                padding='8px',
                                                max_height='400px',
                                                overflow='auto'))
version_row = widgets.HBox(v_buttons)
buttons_row = widgets.HBox([talk_btn, close_btn])
dashboard   = widgets.VBox([title, status,
                            widgets.HTML("<b>Choose a prompt version:</b>"), version_row,
                            client_in, buttons_row, chat_out])

# ============================================================
# CALLBACKS
# ============================================================
def on_talk(_):
    prompt = client_in.value.strip()
    if not prompt:
        return
    client_in.value = ''
    context.append({'role':'user', 'content': prompt})
    response = continue_conversation(context)
    context.append({'role':'assistant', 'content': response})
    with chat_out:
        display(Markdown(f"**User:** {prompt}"))
        display(Markdown(f"**Assistant:** {response}"))

def on_close(_):
    with chat_out:
        clear_output()
    client_in.disabled = True
    talk_btn.disabled  = True
    close_btn.disabled = True
    for b in v_buttons:
        b.disabled = True
    dashboard.children = [widgets.HTML("<h3>👋 Session ended. Thanks for trying the OrderBot!</h3>")]

def make_switch_handler(name):
    def handler(_):
        current_version["name"] = name
        # Reset context to the chosen system prompt — fresh conversation.
        context.clear()
        context.append({'role': 'system', 'content': PROMPTS[name]})
        status.value = f"<b>Active prompt:</b> {name} <i>(conversation reset)</i>"
        with chat_out:
            clear_output()
            display(Markdown(f"*— Switched to **{name}**. Conversation reset. —*"))
    return handler

talk_btn.on_click(on_talk)
close_btn.on_click(on_close)
for btn, name in zip(v_buttons, PROMPTS.keys()):
    btn.on_click(make_switch_handler(name))

display(dashboard)

# Exercise
 - Complete the prompts similar to what we did in class. 
     - Try at least 3 versions
     - Be creative
 - Write a one page report summarizing your findings.
     - Were there variations that didn't work well? i.e., where GPT either hallucinated or wrong
 - What did you learn?

# Report — Findings from 3 Prompt Variations

## Setup
Three system prompts were built for the same OrderBot pattern and exposed as buttons in the GUI so the conversation can be reset and compared on the fly.

| Version | Domain | Style | Notable rules |
|---|---|---|---|
| **V1 · Frankfurt 🌭** | German fast food | Short, friendly | Loose — sizes implied by 3 prices |
| **V2 · Pizzeria 🍕** | Italian pizzeria | Warm, "Ciao!" | Strict size + crust confirmation, EUR, no-invent rule |
| **V3 · Sushi 🍣**   | Japanese sushi  | Polite, Japanese flavor | Allergy check **before** finalizing, USD, strict menu |

## What worked well
- **Explicit workflow steps** (V3's numbered list) produced the most consistent flow — the model rarely skipped the allergy step.
- **"Never invent menu items"** in V2 and V3 reduced hallucinated dishes dramatically vs. V1.
- **Strict size/option enumeration** ("Small / Medium / Large", "Mild / Medium / Hot") forced the model to ask clarifying questions instead of guessing.
- **Tone instructions** ("Ciao!", "Irasshaimase!") were honored consistently and made each bot feel distinct.

## What didn't work / hallucinations observed
- **V1 — ambiguous prices.** The original `burger 12.95, 10.00, 7.00` format has no labels, so GPT sometimes invented size names ("Large/Medium/Small") and sometimes interpreted them as variants ("classic/cheese/mini"). Fixed in V2/V3 by labeling sizes explicitly.
- **Off-menu requests.** Without an explicit *no-invent* rule, GPT happily added a "Hawaiian pizza" or "tempura roll" to V1-style prompts. V2/V3 refused correctly thanks to the rule.
- **Currency drift.** V1 omits currency, so totals appeared as `$`, `€`, or just numbers across runs. Fixing currency in the prompt removed the drift.
- **Order-of-questions slippage.** Even with a numbered workflow, the model sometimes summarized the order *before* asking about allergies. Putting "BEFORE finalizing" in caps helped.
- **Memory of context.** Since the conversation is the full `messages` list, switching prompt versions without clearing the list leaked the old persona — that's why the version buttons reset `context` entirely.

## What I learned
1. **Specificity beats verbosity.** Short prompts with labeled fields (size, currency, crust) outperform long descriptive prose.
2. **Negative constraints matter.** Telling the model what *not* to do ("never invent items") is as important as telling it what to do.
3. **System prompt = persona + rules + menu.** Keeping these three blocks separate makes prompts easier to iterate.
4. **Conversation memory is just the `messages` array.** "Memory" isn't magic — switching personas requires explicitly clearing/replacing the system message and prior turns.
5. **Temperature 0** gives reproducible behavior, which made comparing prompt variations meaningful. Higher temperature reintroduces hallucinations.
6. **GUI feedback loop.** Having buttons to swap prompts in-place made A/B testing the variations far faster than editing code each time.